## Alpha sweep — finding the optimal distance metric

So far we ran all distance configs at **neutral alpha** — the value that gives each feature equal weight, mimicking Gower. This is a fair starting point for comparison, but it is an arbitrary one.

Alpha controls how much the numerical block contributes vs the categorical block:
- `alpha = 1.0` → pure numerical distance, categoricals ignored
- `alpha = 0.0` → pure categorical distance, numericals ignored  
- `alpha = 0.8` → numericals dominate (our neutral baseline for Hamming)
- `alpha = 0.58` → numericals still dominate but less so (our neutral baseline for Tanimoto)

The neutral baseline just says *"I have no prior belief about which block matters more, so I weight them proportionally to how many features each has."* That is a reasonable default but it is not necessarily the best for your data.

The problem with only comparing configs at neutral alpha is that a metric which looks worse than another at neutral alpha might actually be better at a different alpha. For example L2+Hamming might beat L1+Tanimoto if you give the categorical block more weight — we simply do not know yet.

So instead of picking a winner at neutral alpha and then tuning it, we sweep alpha across all configs simultaneously. Every config gets its best possible alpha, and we compare them at their optimal. This is the only fair comparison.

We also run each combination across multiple random seeds and average the silhouette, because with only 49 observations a single kmedoids run can get lucky or unlucky on the initial medoid placement. A difference of 0.02 in silhouette on one seed is noise; a consistent difference across 5 seeds is signal.

The output will be a table of `(num_dist, cat_dist, best_alpha, best_k, mean_silhouette)` — the winner is the row with the highest mean silhouette.

In [21]:
import numpy as np
import pandas as pd
import pickle
import logging
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.ensemble import IsolationForest
import gower as gower_lib
import kmedoids
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import umap
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import sys
sys.path.append(str(Path.cwd()))
from distance import (
    mixed_distance_matrix,
    check_distance_matrix,
    build_gower_cat_mask,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(levelname)s — %(message)s",
)
logger = logging.getLogger(__name__)

In [22]:
df_metadata = pd.read_excel(
    "C:/Users/giuli/Repositories/fintech-group-work/BusinessCase1/Data/BankClients_Metadata.xlsx"
)
print(f"Metadata: {df_metadata.shape[0]:,} rows × {df_metadata.shape[1]} columns")
df_metadata.head()

Metadata: 18 rows × 2 columns


,Metadata,Unnamed: 1
0,ID,Numerical ID
1,Age,"Age, in years"
2,Gender,"Gender (Female = 1, Male = 0)"
3,Job,1 = Unemployed\n2 = Employee/Worker\n3 = Manag...
4,Area,"1 = North, 2 = Center, 3 = South/Islands"


In [23]:
df_raw = pd.read_excel(
    r"C:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\Data\Dataset1_BankClients.xlsx",
    nrows=500  # FOR DEBUGGING, ONLY FIRST 50 INSTANCES
)
print(f"Raw data: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
df_raw.head()


Raw data: 500 rows × 18 columns


,ID,Age,Gender,Job,Area,CitySize,FamilySize,Income,Wealth,Debt,FinEdu,ESG,Digital,BankFriend,LifeStyle,Luxury,Saving,Investments
0,1,24,1,1,2,2,4,0.668046,0.702786,0.262070,0.741853,0.483684,0.698625,0.618259,0.607877,0.897369,0.283222,1
1,2,47,1,2,2,3,1,0.858453,0.915043,0.730430,0.859423,0.537167,0.959025,0.785936,0.862271,0.913729,0.821590,3
2,3,38,0,2,1,2,2,0.926818,0.898316,0.441272,0.485953,0.649434,0.750265,0.699725,0.755404,0.765199,0.503790,3
3,4,67,0,2,1,2,3,0.538797,0.423180,0.600401,0.493144,0.533829,0.590165,0.675353,0.334432,0.517209,0.691240,2
4,5,33,0,2,1,3,1,0.806659,0.731404,0.831449,0.856286,0.784940,0.710026,0.758793,0.908878,0.611610,0.615916,2


In [24]:
# ── Column definitions ───────────────────────────────────────────────────────
categorical_cols = ["Gender", "Job", "Area"]
numerical_cols   = [c for c in df_raw.columns if c not in categorical_cols + ["ID"]]
 
logger.info("Numerical columns (%d): %s", len(numerical_cols), numerical_cols)
logger.info("Categorical columns (%d): %s", len(categorical_cols), categorical_cols)
 
# ── 1. Drop ID ───────────────────────────────────────────────────────────────
df = df_raw.drop(columns=["ID"])
 
# ── 2. Missing values ────────────────────────────────────────────────────────
for col in numerical_cols:
    n_missing = df[col].isna().sum()
    if n_missing:
        logger.info("Imputing %d missing values in '%s' with median.", n_missing, col)
    df[col] = df[col].fillna(df[col].median())
 
for col in categorical_cols:
    n_missing = df[col].isna().sum()
    if n_missing:
        logger.info("Imputing %d missing values in '%s' with mode.", n_missing, col)
    df[col] = df[col].fillna(df[col].mode()[0])
 
assert df.isna().sum().sum() == 0, "Missing values remain after imputation!"
logger.info("Missing values after imputation: 0")
 
# ── 3. Hard domain filter — working minors ───────────────────────────────────
mask_minors = (df["Age"] < 18) & (df["Job"].isin([2, 3, 4]))
n_removed   = mask_minors.sum()
df = df[~mask_minors].reset_index(drop=True)
logger.info("Minor filter: removed %d rows → %d remaining.", n_removed, len(df))
 
# ── 4. Isolation Forest — remove top 1%% multivariate outliers ───────────────
iso    = IsolationForest(contamination=0.01, random_state=42)
labels = iso.fit_predict(df[numerical_cols])
n_out  = (labels == -1).sum()
df     = df[labels == 1].reset_index(drop=True)
logger.info("Isolation Forest: removed %d outliers → %d remaining.", n_out, len(df))
 
# ── 5. Prepare input arrays ──────────────────────────────────────────────────
 
# Numerical: min-max scale to [0, 1]
scaler = MinMaxScaler()
X_num  = scaler.fit_transform(df[numerical_cols]).astype(float)
logger.info("X_num scaled to [0,1]: min=%.4f  max=%.4f", X_num.min(), X_num.max())
 
# Categorical — label-encoded (for Hamming; one integer per variable)
le_encoders = {}
X_cat_int   = np.zeros((len(df), len(categorical_cols)), dtype=np.int32)
for j, col in enumerate(categorical_cols):
    le = LabelEncoder()
    X_cat_int[:, j] = le.fit_transform(df[col])
    le_encoders[col] = le
    logger.info("LabelEncoder '%s': %d classes → %s", col, len(le.classes_), list(le.classes_))
 
# Categorical — one-hot encoded (for Tanimoto; one binary column per category level)
X_cat_ohe = pd.get_dummies(df[categorical_cols].astype(str)).values.astype(np.int32)
logger.info("One-hot encoded shape: %s  unique values: %s",
            X_cat_ohe.shape, np.unique(X_cat_ohe).tolist())
 
 
n_num = X_num.shape[1]

# Hamming: n_cat = number of original categorical variables (one column each)
alpha_neutral_hamming  = n_num / (n_num + X_cat_int.shape[1])
# Tanimoto: n_cat = number of one-hot binary columns (one per category level)
alpha_neutral_tanimoto = n_num / (n_num + X_cat_ohe.shape[1])

logger.info(
    "Neutral alpha (Hamming)  = %d / (%d + %d) = %.4f",
    n_num, n_num, X_cat_int.shape[1], alpha_neutral_hamming,
)
logger.info(
    "Neutral alpha (Tanimoto) = %d / (%d + %d) = %.4f",
    n_num, n_num, X_cat_ohe.shape[1], alpha_neutral_tanimoto,
)
 
print(f"\nX_num shape     : {X_num.shape}")
print(f"X_cat_int shape : {X_cat_int.shape}  unique={np.unique(X_cat_int).tolist()}")
print(f"X_cat_ohe shape : {X_cat_ohe.shape}  unique={np.unique(X_cat_ohe).tolist()}")
print(f"Neutral alpha (Hamming)  : {alpha_neutral_hamming:.4f}")
print(f"Neutral alpha (Tanimoto) : {alpha_neutral_tanimoto:.4f}")

2026-03-19 20:47:22,896 — INFO — Numerical columns (14): ['Age', 'CitySize', 'FamilySize', 'Income', 'Wealth', 'Debt', 'FinEdu', 'ESG', 'Digital', 'BankFriend', 'LifeStyle', 'Luxury', 'Saving', 'Investments']
2026-03-19 20:47:22,898 — INFO — Categorical columns (3): ['Gender', 'Job', 'Area']
2026-03-19 20:47:22,923 — INFO — Missing values after imputation: 0
2026-03-19 20:47:22,928 — INFO — Minor filter: removed 0 rows → 500 remaining.
2026-03-19 20:47:23,137 — INFO — Isolation Forest: removed 5 outliers → 495 remaining.
2026-03-19 20:47:23,144 — INFO — X_num scaled to [0,1]: min=0.0000  max=1.0000
2026-03-19 20:47:23,146 — INFO — LabelEncoder 'Gender': 2 classes → [np.int64(0), np.int64(1)]
2026-03-19 20:47:23,146 — INFO — LabelEncoder 'Job': 5 classes → [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
2026-03-19 20:47:23,147 — INFO — LabelEncoder 'Area': 3 classes → [np.int64(1), np.int64(2), np.int64(3)]
2026-03-19 20:47:23,162 — INFO — One-hot encoded shape: (495, 


X_num shape     : (495, 14)
X_cat_int shape : (495, 3)  unique=[0, 1, 2, 3, 4]
X_cat_ohe shape : (495, 10)  unique=[0, 1]
Neutral alpha (Hamming)  : 0.8235
Neutral alpha (Tanimoto) : 0.5833


In [25]:
CACHE_DIR = Path("distance_cache")
CACHE_DIR.mkdir(exist_ok=True)
WEIGHTED_CACHE_DIR = CACHE_DIR / "weighted"
WEIGHTED_CACHE_DIR.mkdir(exist_ok=True)

def load_or_compute(path: Path, compute_fn):
    """Load matrix from pickle if it exists, otherwise compute and save."""
    if path.exists():
        logger.info("Loading cached matrix from %s", path)
        with open(path, "rb") as f:
            return pickle.load(f)
    logger.info("Computing matrix — will save to %s", path)
    D = compute_fn()
    with open(path, "wb") as f:
        pickle.dump(D, f)
    logger.info("Saved to %s", path)
    return D

# ── Mixed configs ─────────────────────────────────────────────────────────────
mixed_configs = [
    ("L1",       "Hamming",  X_cat_int.copy(), alpha_neutral_hamming),
    ("L1",       "Tanimoto", X_cat_ohe.copy(), alpha_neutral_tanimoto),
    ("L2",       "Hamming",  X_cat_int.copy(), alpha_neutral_hamming),
    ("L2",       "Tanimoto", X_cat_ohe.copy(), alpha_neutral_tanimoto),
    ("Canberra", "Hamming",  X_cat_int.copy(), alpha_neutral_hamming),
    ("Canberra", "Tanimoto", X_cat_ohe.copy(), alpha_neutral_tanimoto),
]

# ── Mixed matrices ────────────────────────────────────────────────────────────
mixed_matrices = {}
for num_dist, cat_dist, X_cat, alpha in mixed_configs:
    key      = f"Mixed ({num_dist}+{cat_dist})"
    pkl_name = f"D_mixed_{num_dist.lower()}_{cat_dist.lower()}.pkl"
    mixed_matrices[key] = load_or_compute(
        WEIGHTED_CACHE_DIR / pkl_name,
        lambda nd=num_dist, cd=cat_dist, xc=X_cat, a=alpha: mixed_distance_matrix(
            X_num, xc,
            alpha=a,
            num_dist=nd,
            cat_dist=cd,
        )
    )

# ── Gower (library) ───────────────────────────────────────────────────────────
cat_mask = build_gower_cat_mask(df, categorical_cols=categorical_cols)

D_gower = load_or_compute(
    WEIGHTED_CACHE_DIR / "D_gower.pkl",
    lambda: gower_lib.gower_matrix(df.values, cat_features=cat_mask)
)
logger.info("Gower matrix ready — shape=%s  min=%.4f  max=%.4f",
            D_gower.shape, D_gower.min(), D_gower.max())

# ── Summary ───────────────────────────────────────────────────────────────────
all_matrices = {**mixed_matrices, "Gower": D_gower}

print("\nAll matrices ready:")
for key, D in all_matrices.items():
    print(f"  {key:35s}: {D.shape}  min={D.min():.4f}  max={D.max():.4f}")

2026-03-19 20:47:23,189 — INFO — Computing matrix — will save to distance_cache\weighted\D_mixed_l1_hamming.pkl


2026-03-19 20:47:23,194 — INFO — ── Pre-flight check: numerical 'X_num[L1+Hamming]'  shape=(495, 14) ──
2026-03-19 20:47:23,195 — INFO — [X_num[L1+Hamming]] OK   dtype: float64
2026-03-19 20:47:23,197 — INFO — [X_num[L1+Hamming]] OK   NaN: none
2026-03-19 20:47:23,197 — INFO — [X_num[L1+Hamming]] OK   Inf: none
2026-03-19 20:47:23,197 — INFO — [X_num[L1+Hamming]] OK   range [0,1]: global min=0.000000  max=1.000000
2026-03-19 20:47:23,199 — INFO — [X_num[L1+Hamming]] OK   constant: no constant columns
2026-03-19 20:47:23,200 — INFO — [X_num[L1+Hamming]] numerical pre-flight: ALL CHECKS PASSED (14 features)
2026-03-19 20:47:23,201 — INFO — ── Pre-flight check: label-encoded categorical 'X_cat[L1+Hamming]'  shape=(495, 3) ──
2026-03-19 20:47:23,201 — INFO — [X_cat[L1+Hamming]] OK   dtype: int32
2026-03-19 20:47:23,203 — INFO — [X_cat[L1+Hamming]] OK   negative codes: none
2026-03-19 20:47:23,203 — INFO — [X_cat[L1+Hamming]] OK   col 0: 2 unique values [0, 1]
2026-03-19 20:47:23,204 — INFO


All matrices ready:
  Mixed (L1+Hamming)                 : (495, 495)  min=0.0000  max=0.6984
  Mixed (L1+Tanimoto)                : (495, 495)  min=0.0000  max=0.7864
  Mixed (L2+Hamming)                 : (495, 495)  min=0.0000  max=0.7335
  Mixed (L2+Tanimoto)                : (495, 495)  min=0.0000  max=0.8113
  Mixed (Canberra+Hamming)           : (495, 495)  min=0.0000  max=0.7578
  Mixed (Canberra+Tanimoto)          : (495, 495)  min=0.0000  max=0.8284
  Gower                              : (495, 495)  min=0.0000  max=0.6984


In [26]:
logger.info("=" * 60)
logger.info("SANITY CHECKS — all distance matrices")
logger.info("=" * 60)

check_results = {}
for name, D in all_matrices.items():
    check_results[name] = check_distance_matrix(D, name=name)

# Summary table
print("\n── Sanity check summary ──")
print(f"{'Matrix':<35} {'Pass?'}")
print("-" * 42)
for name, ok in check_results.items():
    status = "PASS" if ok else "FAIL"
    print(f"{name:<35} {status}")

# Cross-check: L1+Hamming vs Gower (should be identical at neutral alpha)
idx       = np.triu_indices(49, k=1)
corr      = np.corrcoef(all_matrices["Mixed (L1+Hamming)"][idx], all_matrices["Gower"][idx])[0, 1]
max_diff  = np.abs(all_matrices["Mixed (L1+Hamming)"][idx] - all_matrices["Gower"][idx]).max()
mean_diff = np.abs(all_matrices["Mixed (L1+Hamming)"][idx] - all_matrices["Gower"][idx]).mean()
logger.info(
    "L1+Hamming vs Gower | correlation=%.8f  mean_abs_diff=%.2e  max_abs_diff=%.2e",
    corr, mean_diff, max_diff,
)
print(f"\n── L1+Hamming vs Gower equivalence check ──")
print(f"  Pearson correlation : {corr:.8f}  (expect 1.0)")
print(f"  Mean abs difference : {mean_diff:.2e}  (expect ~0)")
print(f"  Max  abs difference : {max_diff:.2e}  (expect ~0)")

# Per-matrix pairwise distance statistics
print("\n── Pairwise distance statistics (upper triangle) ──")
header = f"{'Matrix':<35} {'mean':>8} {'std':>8} {'p25':>8} {'p50':>8} {'p75':>8} {'max':>8}"
print(header)
print("-" * len(header))
for name, D in all_matrices.items():
    vals = D[np.triu_indices(D.shape[0], k=1)]
    print(
        f"{name:<35} {vals.mean():8.4f} {vals.std():8.4f} "
        f"{np.percentile(vals,25):8.4f} {np.percentile(vals,50):8.4f} "
        f"{np.percentile(vals,75):8.4f} {vals.max():8.4f}"
    )

2026-03-19 20:48:12,920 — INFO — ============================================================
2026-03-19 20:48:12,921 — INFO — SANITY CHECKS — all distance matrices
2026-03-19 20:48:12,922 — INFO — ============================================================
2026-03-19 20:48:12,923 — INFO — ── Checking distance matrix: Mixed (L1+Hamming)  shape=(495, 495) ──
2026-03-19 20:48:12,925 — INFO — [Mixed (L1+Hamming)] OK   diagonal: 0.00e+00
2026-03-19 20:48:12,927 — INFO — [Mixed (L1+Hamming)] OK   symmetry: 0.00e+00
2026-03-19 20:48:12,929 — INFO — [Mixed (L1+Hamming)] OK   non-negativity: min=0.000000
2026-03-19 20:48:12,931 — INFO — [Mixed (L1+Hamming)] OK   range: [0.000000, 0.698409]
2026-03-19 20:48:12,932 — INFO — [Mixed (L1+Hamming)] OK   NaN/Inf: none
2026-03-19 20:48:12,940 — INFO — [Mixed (L1+Hamming)] OK   off-diagonal: mean=0.3222  ~0 frac=0.0%
2026-03-19 20:48:12,942 — INFO — [Mixed (L1+Hamming)] ALL CHECKS PASSED
2026-03-19 20:48:12,942 — INFO — ── Checking distance matrix: Mi


── Sanity check summary ──
Matrix                              Pass?
------------------------------------------
Mixed (L1+Hamming)                  PASS
Mixed (L1+Tanimoto)                 PASS
Mixed (L2+Hamming)                  PASS
Mixed (L2+Tanimoto)                 PASS
Mixed (Canberra+Hamming)            PASS
Mixed (Canberra+Tanimoto)           PASS
Gower                               PASS

── L1+Hamming vs Gower equivalence check ──
  Pearson correlation : 1.00000000  (expect 1.0)
  Mean abs difference : 7.69e-09  (expect ~0)
  Max  abs difference : 3.58e-08  (expect ~0)

── Pairwise distance statistics (upper triangle) ──
Matrix                                  mean      std      p25      p50      p75      max
-----------------------------------------------------------------------------------------
Mixed (L1+Hamming)                    0.3222   0.0873   0.2614   0.3204   0.3807   0.6984
Mixed (L1+Tanimoto)                   0.4147   0.1355   0.3456   0.4288   0.5145   0.7864
M

In [27]:
from itertools import product

ALPHA_SWEEP   = [0.2, 0.3, 0.4, 0.5, 0.6, alpha_neutral_hamming, alpha_neutral_tanimoto, 0.8, 0.9]
ALPHA_SWEEP   = sorted(set(round(a, 4) for a in ALPHA_SWEEP))  # deduplicate & sort
K_RANGE       = range(2, 11)
SEEDS         = [42, 7, 123, 999, 2024]

# configs without alpha — we sweep it here
sweep_configs = [
    ("L1",       "Hamming",  X_cat_int.copy()),
    ("L1",       "Tanimoto", X_cat_ohe.copy()),
    ("L2",       "Hamming",  X_cat_int.copy()),
    ("L2",       "Tanimoto", X_cat_ohe.copy()),
    ("Canberra", "Hamming",  X_cat_int.copy()),
    ("Canberra", "Tanimoto", X_cat_ohe.copy()),
]

logger.info("Alpha sweep: %s", ALPHA_SWEEP)
logger.info("Seeds: %s", SEEDS)
logger.info("Configs: %d  ×  alphas: %d  ×  k values: %d  ×  seeds: %d  =  %d total runs",
            len(sweep_configs), len(ALPHA_SWEEP), len(K_RANGE),
            len(SEEDS), len(sweep_configs) * len(ALPHA_SWEEP) * len(K_RANGE) * len(SEEDS))

sweep_results = []

for num_dist, cat_dist, X_cat in sweep_configs:
    for alpha in ALPHA_SWEEP:
        # compute distance matrix for this (metric, alpha) combination
        D = mixed_distance_matrix(
            X_num, X_cat,
            alpha=alpha,
            num_dist=num_dist,
            cat_dist=cat_dist,
        ).astype(np.float64)

        for k in K_RANGE:
            sil_scores = []
            for seed in SEEDS:
                res  = kmedoids.fasterpam(D, k, random_state=seed)
                lbls = np.array(res.labels)
                # need at least 2 clusters with >1 member for silhouette
                if len(np.unique(lbls)) < 2:
                    continue
                sil_scores.append(silhouette_score(D, lbls, metric="precomputed"))

            if not sil_scores:
                continue

            mean_sil = float(np.mean(sil_scores))
            std_sil  = float(np.std(sil_scores))

            sweep_results.append({
                "num_dist": num_dist,
                "cat_dist": cat_dist,
                "alpha":    round(alpha, 4),
                "k":        k,
                "mean_sil": round(mean_sil, 4),
                "std_sil":  round(std_sil,  4),
            })

df_sweep = pd.DataFrame(sweep_results)

# best alpha+k per config (highest mean silhouette)
df_best = (
    df_sweep
    .sort_values("mean_sil", ascending=False)
    .groupby(["num_dist", "cat_dist"], sort=False)
    .first()
    .reset_index()
    [["num_dist", "cat_dist", "alpha", "k", "mean_sil", "std_sil"]]
    .sort_values("mean_sil", ascending=False)
    .reset_index(drop=True)
)

print("\n── Best (alpha, k) per distance config ──")
print(df_best.to_string(index=False))

# ── Gower reference — sweep k directly since results dict does not exist ──────
D_gower_f64 = all_matrices["Gower"].astype(np.float64)
gower_sil_rows = []

for k in K_RANGE:
    sil_scores = []
    for seed in SEEDS:
        res  = kmedoids.fasterpam(D_gower_f64, k, random_state=seed)
        lbls = np.array(res.labels)
        if len(np.unique(lbls)) >= 2:
            sil_scores.append(silhouette_score(D_gower_f64, lbls, metric="precomputed"))
    if sil_scores:
        gower_sil_rows.append({"k": k, "mean_sil": np.mean(sil_scores), "std_sil": np.std(sil_scores)})

best_gower = max(gower_sil_rows, key=lambda r: r["mean_sil"])
gower_sil  = [r["mean_sil"] for r in gower_sil_rows if r["k"] == best_gower["k"]]

print(f"\nGower (best k={best_gower['k']})  "
      f"mean_sil={best_gower['mean_sil']:.4f}  std={best_gower['std_sil']:.4f}")

2026-03-19 20:48:13,083 — INFO — Alpha sweep: [0.2, 0.3, 0.4, 0.5, 0.5833, 0.6, 0.8, 0.8235, 0.9]
2026-03-19 20:48:13,085 — INFO — Seeds: [42, 7, 123, 999, 2024]
2026-03-19 20:48:13,086 — INFO — Configs: 6  ×  alphas: 9  ×  k values: 9  ×  seeds: 5  =  2430 total runs
2026-03-19 20:48:13,099 — INFO — ── Pre-flight check: numerical 'X_num[L1+Hamming]'  shape=(495, 14) ──
2026-03-19 20:48:13,100 — INFO — [X_num[L1+Hamming]] OK   dtype: float64
2026-03-19 20:48:13,103 — INFO — [X_num[L1+Hamming]] OK   NaN: none
2026-03-19 20:48:13,104 — INFO — [X_num[L1+Hamming]] OK   Inf: none
2026-03-19 20:48:13,104 — INFO — [X_num[L1+Hamming]] OK   range [0,1]: global min=0.000000  max=1.000000
2026-03-19 20:48:13,105 — INFO — [X_num[L1+Hamming]] OK   constant: no constant columns
2026-03-19 20:48:13,106 — INFO — [X_num[L1+Hamming]] numerical pre-flight: ALL CHECKS PASSED (14 features)
2026-03-19 20:48:13,106 — INFO — ── Pre-flight check: label-encoded categorical 'X_cat[L1+Hamming]'  shape=(495, 3) ──


── Best (alpha, k) per distance config ──
num_dist cat_dist  alpha  k  mean_sil  std_sil
      L1 Tanimoto    0.2 10    0.6924      0.0
Canberra Tanimoto    0.2 10    0.6798      0.0
      L2 Tanimoto    0.2 10    0.6715      0.0
      L1  Hamming    0.2 10    0.6574      0.0
Canberra  Hamming    0.2 10    0.6426      0.0
      L2  Hamming    0.2 10    0.6286      0.0

Gower (best k=2)  mean_sil=0.1774  std=0.0271


In [28]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── 1. Silhouette heatmap: alpha × k, one subplot per config ─────────────────
configs_ordered = [f"{nd}+{cd}" for nd, cd, _ in sweep_configs]
n_configs = len(configs_ordered)

fig_heat = make_subplots(
    rows=2, cols=3,
    subplot_titles=configs_ordered,
    horizontal_spacing=0.08,
    vertical_spacing=0.14,
)

for idx, (num_dist, cat_dist, _) in enumerate(sweep_configs):
    row = idx // 3 + 1
    col = idx %  3 + 1

    subset = df_sweep[
        (df_sweep["num_dist"] == num_dist) &
        (df_sweep["cat_dist"] == cat_dist)
    ]
    pivot = subset.pivot(index="alpha", columns="k", values="mean_sil")

    fig_heat.add_trace(
        go.Heatmap(
            z=pivot.values,
            x=[str(k) for k in pivot.columns],
            y=[str(round(a, 2)) for a in pivot.index],
            colorscale="Viridis",
            zmin=df_sweep["mean_sil"].min(),
            zmax=df_sweep["mean_sil"].max(),
            showscale=(idx == 0),
            colorbar=dict(title="Silhouette", x=1.02) if idx == 0 else None,
        ),
        row=row, col=col,
    )
    fig_heat.update_xaxes(title_text="k", row=row, col=col)
    fig_heat.update_yaxes(title_text="alpha", row=row, col=col)

fig_heat.update_layout(
    title="Mean silhouette — alpha × k heatmap per config",
    height=700,
)
fig_heat.show()

# ── 2. Best silhouette per config — bar chart with std error bars ─────────────
fig_bar = go.Figure()

colors = px.colors.qualitative.Vivid[:len(df_best)]

fig_bar.add_trace(go.Bar(
    x=df_best["num_dist"] + "+" + df_best["cat_dist"],
    y=df_best["mean_sil"],
    error_y=dict(type="data", array=df_best["std_sil"].tolist(), visible=True),
    marker_color=colors,
    text=[
        f"α={row.alpha:.2f}<br>k={row.k}"
        for row in df_best.itertuples()
    ],
    textposition="outside",
))

# Gower reference line
fig_bar.add_hline(
    y=float(np.mean(gower_sil)),
    line_dash="dash",
    line_color="gray",
    annotation_text=f"Gower ({np.mean(gower_sil):.3f})",
    annotation_position="top left",
)

fig_bar.update_layout(
    title="Best mean silhouette per config (at optimal alpha and k)",
    yaxis_title="Mean silhouette (5 seeds)",
    xaxis_title="Distance config",
    height=500,
    showlegend=False,
)
fig_bar.show()

# ── 3. Silhouette vs alpha curves — one line per config ───────────────────────
# collapse over k: for each (config, alpha) take the best k
df_alpha_curve = (
    df_sweep
    .sort_values("mean_sil", ascending=False)
    .groupby(["num_dist", "cat_dist", "alpha"])
    .first()
    .reset_index()
)

fig_curve = go.Figure()

for num_dist, cat_dist, _ in sweep_configs:
    sub = df_alpha_curve[
        (df_alpha_curve["num_dist"] == num_dist) &
        (df_alpha_curve["cat_dist"] == cat_dist)
    ].sort_values("alpha")

    fig_curve.add_trace(go.Scatter(
        x=sub["alpha"],
        y=sub["mean_sil"],
        mode="lines+markers",
        name=f"{num_dist}+{cat_dist}",
        error_y=dict(
            type="data",
            array=sub["std_sil"].tolist(),
            visible=True,
            thickness=1,
            width=4,
        ),
    ))

# neutral alpha reference lines
fig_curve.add_vline(
    x=alpha_neutral_hamming,
    line_dash="dot", line_color="gray",
    annotation_text=f"neutral Hamming ({alpha_neutral_hamming:.2f})",
    annotation_position="top right",
)
fig_curve.add_vline(
    x=alpha_neutral_tanimoto,
    line_dash="dot", line_color="lightgray",
    annotation_text=f"neutral Tanimoto ({alpha_neutral_tanimoto:.2f})",
    annotation_position="bottom right",
)

fig_curve.update_layout(
    title="Silhouette vs alpha — best k at each alpha",
    xaxis_title="alpha (numerical block weight)",
    yaxis_title="Mean silhouette (5 seeds)",
    height=520,
    legend=dict(x=0.01, y=0.99),
)
fig_curve.show()

## Alpha sweep — silhouette vs alpha curve

Every config shows a monotonically decreasing silhouette as alpha increases — clustering quality consistently improves as more weight is given to the categorical block. The optimal alpha for all six configs is at the leftmost point tested (alpha=0.2), meaning the categorical variables dominate the best-performing distance in every case.

Several observations worth noting:

**Tanimoto dominates Hamming at every alpha.** The gap between the two encoding families is consistent across the full sweep, not just at neutral alpha. This rules out the possibility that Tanimoto's advantage was an artefact of the neutral baseline — it holds regardless of weighting.

**L1+Tanimoto is the best config at every alpha.** It sits above all other curves for the entire range, confirming it as the primary candidate.

**All configs converge toward alpha=0.82–0.84 (neutral Hamming).** At the neutral Tanimoto baseline (0.58, dashed line) the spread between configs is already narrow, and by alpha=0.82 nearly all configs produce equivalent, poor silhouette scores around 0.15–0.20. This is consistent with the not-weighted results, where neutral alpha produced weak structure.

**The error bars at alpha=0.58–0.60 are visible only for L1+Tanimoto and L2+Tanimoto**, indicating those configs have slightly more variance across seeds in that region. All other configs are stable throughout.

### Conclusion

The sweep is unambiguous: categorical features drive the cluster structure in this dataset. The weighted notebook will select alpha=0.2 as optimal, giving the categorical block 80% of the distance weight. This is not a marginal improvement — silhouette at alpha=0.2 (~0.65) is more than three times higher than at the neutral Hamming baseline (~0.19).

In [29]:
# ── Build best labels from sweep results ──────────────────────────────────────
# For each config, re-run kmedoids at the best (alpha, k) found in the sweep
# using the majority-vote seed (seed=42) for reproducibility
umap_sources = {}

for _, row in df_best.iterrows():
    name     = f"Mixed ({row.num_dist}+{row.cat_dist})"
    best_k     = row.k
    best_alpha = row.alpha
    X_cat    = next(xc for nd, cd, xc in sweep_configs
                    if nd == row.num_dist and cd == row.cat_dist)

    D = mixed_distance_matrix(
        X_num, X_cat,
        alpha=best_alpha,
        num_dist=row.num_dist,
        cat_dist=row.cat_dist,
    ).astype(np.float64)

    res  = kmedoids.fasterpam(D, best_k, random_state=42)
    lbls = np.array(res.labels)
    umap_sources[name] = {"D": D, "labels": lbls, "k": best_k, "alpha": best_alpha}

# Gower
D_gower_f64 = all_matrices["Gower"].astype(np.float64)
res_gower   = kmedoids.fasterpam(D_gower_f64, best_gower["k"], random_state=42)
umap_sources["Gower"] = {
    "D":      D_gower_f64,
    "labels": np.array(res_gower.labels),
    "k":      best_gower["k"],
    "alpha":  None,
}

# ── UMAP plots ────────────────────────────────────────────────────────────────
umap_embeddings = {}

for name, src in umap_sources.items():
    D   = src["D"]
    lbl = src["labels"]
    k   = src["k"]
    alpha_tag = f"  α={src['alpha']:.2f}" if src["alpha"] is not None else ""

    reducer_2d = umap.UMAP(n_components=2, metric="precomputed",
                            n_neighbors=8, min_dist=0.1, random_state=42)
    emb2d = reducer_2d.fit_transform(D)

    reducer_3d = umap.UMAP(n_components=3, metric="precomputed",
                            n_neighbors=8, min_dist=0.1, random_state=42)
    emb3d = reducer_3d.fit_transform(D)

    umap_embeddings[name] = {"2d": emb2d, "3d": emb3d}

    cluster_labels = [f"Cluster {c}" for c in lbl.tolist()]

    df_2d = pd.DataFrame({
        "UMAP1": emb2d[:, 0], "UMAP2": emb2d[:, 1],
        "Cluster": cluster_labels,
    })
    fig2d = px.scatter(
        df_2d, x="UMAP1", y="UMAP2", color="Cluster",
        title=f"UMAP 2D — {name} (k={k}{alpha_tag})",
        opacity=0.6,
        color_discrete_sequence=px.colors.qualitative.Vivid,
    )
    fig2d.update_traces(marker=dict(size=4))
    fig2d.update_layout(height=520)
    fig2d.show()

    df_3d = pd.DataFrame({
        "UMAP1": emb3d[:, 0], "UMAP2": emb3d[:, 1], "UMAP3": emb3d[:, 2],
        "Cluster": cluster_labels,
    })
    fig3d = px.scatter_3d(
        df_3d, x="UMAP1", y="UMAP2", z="UMAP3", color="Cluster",
        title=f"UMAP 3D — {name} (k={k}{alpha_tag})",
        opacity=0.65,
        color_discrete_sequence=px.colors.qualitative.Vivid,
    )
    fig3d.update_traces(marker=dict(size=3))
    fig3d.update_layout(height=620)
    fig3d.show()

2026-03-19 20:55:49,533 — INFO — ── Pre-flight check: numerical 'X_num[L1+Tanimoto]'  shape=(495, 14) ──
2026-03-19 20:55:49,535 — INFO — [X_num[L1+Tanimoto]] OK   dtype: float64
2026-03-19 20:55:49,536 — INFO — [X_num[L1+Tanimoto]] OK   NaN: none
2026-03-19 20:55:49,537 — INFO — [X_num[L1+Tanimoto]] OK   Inf: none
2026-03-19 20:55:49,538 — INFO — [X_num[L1+Tanimoto]] OK   range [0,1]: global min=0.000000  max=1.000000
2026-03-19 20:55:49,540 — INFO — [X_num[L1+Tanimoto]] OK   constant: no constant columns
2026-03-19 20:55:49,540 — INFO — [X_num[L1+Tanimoto]] numerical pre-flight: ALL CHECKS PASSED (14 features)
2026-03-19 20:55:49,543 — INFO — ── Pre-flight check: binary (one-hot) 'X_cat[L1+Tanimoto]'  shape=(495, 10) ──
2026-03-19 20:55:49,544 — INFO — [X_cat[L1+Tanimoto]] OK   binary values: only {0, 1}
2026-03-19 20:55:49,545 — INFO — [X_cat[L1+Tanimoto]] OK   NaN/Inf: none
2026-03-19 20:55:49,545 — INFO — [X_cat[L1+Tanimoto]] OK   column activity: min_freq=0.020  max_freq=0.749  m

c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [30]:
JOB_MAP  = {1: "Unemployed", 2: "Employee", 3: "Manager",
             4: "Entrepreneur", 5: "Retired"}
INV_MAP  = {1: "None", 2: "Lump Sum", 3: "Cap. Accum."}
GEN_MAP  = {0: "Male", 1: "Female"}
AREA_MAP = {1: "North", 2: "Center", 3: "South"}

KEY_FEATURES = ["Income", "Wealth", "Debt", "Saving", "Luxury", "FinEdu"]
PAL          = px.colors.qualitative.Vivid

for name, src in umap_sources.items():
    k   = src["k"]
    lbl = src["labels"]
    alpha_tag = f"  α={src['alpha']:.2f}" if src["alpha"] is not None else ""

    df_p = df.copy()
    df_p["Cluster"] = lbl

    num_profile_cols = [c for c in numerical_cols if c in df_p.columns]

    # ── Summary table ─────────────────────────────────────────────────────────
    rows = []
    for c in range(k):
        g   = df_p[df_p["Cluster"] == c]
        row = {"Cluster": c, "N": len(g), "%": f"{len(g)/len(df_p)*100:.1f}%"}
        for col in num_profile_cols:
            row[col] = round(float(g[col].mean()), 3)
        row["Job"]    = JOB_MAP.get(int(g["Job"].mode()[0]), "?")
        row["Gender"] = GEN_MAP.get(int(g["Gender"].mode()[0]), "?")
        row["Area"]   = AREA_MAP.get(int(g["Area"].mode()[0]), "?")
        rows.append(row)

    print(f"\n{'='*60}")
    print(f" {name} — k={k}{alpha_tag} cluster summary")
    print(f"{'='*60}")
    display(pd.DataFrame(rows))

    # ── Radar chart ───────────────────────────────────────────────────────────
    radar = go.Figure()
    for c in range(k):
        g    = df_p[df_p["Cluster"] == c]
        vals = [float(g[col].mean()) for col in num_profile_cols]
        radar.add_trace(go.Scatterpolar(
            r=vals + [vals[0]],
            theta=num_profile_cols + [num_profile_cols[0]],
            fill="toself", name=f"Cluster {c}",
            line_color=PAL[c], opacity=0.75,
        ))
    radar.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        title=f"Radar — {name} (k={k}{alpha_tag})",
        height=500,
    )
    radar.show()

    # ── Boxplots for key features ─────────────────────────────────────────────
    box = make_subplots(rows=1, cols=len(KEY_FEATURES),
                        subplot_titles=KEY_FEATURES)
    for ci, col in enumerate(KEY_FEATURES):
        for c in range(k):
            g = df_p[df_p["Cluster"] == c]
            box.add_trace(go.Box(
                y=g[col].tolist(), name=f"Cluster {c}",
                marker_color=PAL[c], showlegend=(ci == 0),
            ), row=1, col=ci + 1)
    box.update_layout(
        height=420, boxmode="group",
        title_text=f"Key Features by Cluster — {name} (k={k}{alpha_tag})",
    )
    box.show()


 Mixed (L1+Tanimoto) — k=10  α=0.20 cluster summary


,Cluster,N,%,Age,CitySize,FamilySize,Income,Wealth,Debt,FinEdu,ESG,Digital,BankFriend,LifeStyle,Luxury,Saving,Investments,Job,Gender,Area
0,0,18,3.6%,57.056,1.611,3.056,0.508,0.567,0.526,0.528,0.536,0.526,0.572,0.425,0.423,0.516,2.222,Employee,Male,South
1,1,21,4.2%,69.952,1.762,2.905,0.570,0.574,0.528,0.521,0.602,0.516,0.627,0.428,0.487,0.550,2.000,Employee,Female,South
2,2,123,24.8%,56.317,2.049,2.407,0.593,0.617,0.499,0.550,0.581,0.583,0.611,0.525,0.523,0.582,2.228,Employee,Female,North
3,3,35,7.1%,53.514,1.971,2.743,0.617,0.675,0.490,0.548,0.560,0.587,0.669,0.528,0.556,0.563,2.229,Employee,Male,Center
4,4,23,4.6%,60.826,2.043,2.217,0.571,0.578,0.314,0.494,0.569,0.491,0.636,0.468,0.444,0.390,2.043,Unemployed,Male,North
5,5,129,26.1%,57.705,1.992,2.713,0.626,0.627,0.512,0.571,0.574,0.588,0.637,0.517,0.571,0.597,2.403,Employee,Male,North
6,6,20,4.0%,55.550,1.850,2.600,0.554,0.569,0.479,0.552,0.522,0.584,0.555,0.498,0.476,0.580,2.050,Employee,Female,Center
7,7,47,9.5%,74.936,1.638,2.340,0.451,0.428,0.232,0.410,0.607,0.367,0.685,0.354,0.332,0.374,1.872,Retired,Male,North
8,8,52,10.5%,75.096,1.865,1.846,0.461,0.487,0.229,0.423,0.642,0.345,0.636,0.253,0.358,0.389,2.096,Retired,Female,North
9,9,27,5.5%,67.593,1.667,2.444,0.515,0.482,0.242,0.384,0.555,0.363,0.616,0.349,0.361,0.407,2.000,Unemployed,Female,North



 Mixed (Canberra+Tanimoto) — k=10  α=0.20 cluster summary


,Cluster,N,%,Age,CitySize,FamilySize,Income,Wealth,Debt,FinEdu,ESG,Digital,BankFriend,LifeStyle,Luxury,Saving,Investments,Job,Gender,Area
0,0,18,3.6%,57.056,1.611,3.056,0.508,0.567,0.526,0.528,0.536,0.526,0.572,0.425,0.423,0.516,2.222,Employee,Male,South
1,1,46,9.3%,74.826,1.630,2.370,0.451,0.424,0.229,0.408,0.603,0.373,0.689,0.358,0.329,0.370,1.891,Retired,Male,North
2,2,124,25.1%,56.589,2.040,2.411,0.592,0.615,0.499,0.550,0.580,0.582,0.611,0.522,0.519,0.584,2.226,Employee,Female,North
3,3,20,4.0%,63.950,2.000,2.400,0.559,0.575,0.327,0.492,0.575,0.486,0.644,0.473,0.436,0.363,2.100,Unemployed,Male,North
4,4,22,4.4%,68.182,1.727,3.000,0.570,0.576,0.517,0.526,0.607,0.511,0.626,0.428,0.486,0.541,2.045,Employee,Female,South
5,5,131,26.5%,57.496,2.008,2.687,0.627,0.628,0.507,0.568,0.576,0.588,0.637,0.516,0.570,0.597,2.397,Employee,Male,North
6,6,21,4.2%,53.952,1.810,2.667,0.559,0.573,0.465,0.547,0.527,0.598,0.561,0.512,0.485,0.555,2.000,Employee,Female,Center
7,7,37,7.5%,53.649,1.946,2.649,0.611,0.666,0.482,0.551,0.558,0.568,0.659,0.516,0.551,0.562,2.162,Employee,Male,Center
8,8,51,10.3%,74.804,1.882,1.824,0.461,0.489,0.223,0.420,0.646,0.343,0.636,0.254,0.363,0.382,2.098,Retired,Female,North
9,9,25,5.1%,70.880,1.720,2.280,0.507,0.470,0.242,0.371,0.545,0.341,0.614,0.328,0.344,0.423,2.000,Unemployed,Female,North



 Mixed (L2+Tanimoto) — k=10  α=0.20 cluster summary


...

,Cluster,N,%,Age,CitySize,FamilySize,Income,Wealth,Debt,FinEdu,ESG,Digital,BankFriend,LifeStyle,Luxury,Saving,Investments,Job,Gender,Area
0,0,18,3.6%,57.056,1.611,3.056,0.508,0.567,0.526,0.528,0.536,0.526,0.572,0.425,0.423,0.516,2.222,Employee,Male,South
1,1,20,4.0%,63.950,2.000,2.400,0.559,0.575,0.327,0.492,0.575,0.486,0.644,0.473,0.436,0.363,2.100,Unemployed,Male,North
2,2,124,25.1%,56.589,2.040,2.411,0.592,0.615,0.499,0.550,0.580,0.582,0.611,0.522,0.519,0.584,2.226,Employee,Female,North
3,3,46,9.3%,74.826,1.630,2.370,0.451,0.424,0.229,0.408,0.603,0.373,0.689,0.358,0.329,0.370,1.891,Retired,Male,North
4,4,22,4.4%,68.182,1.727,3.000,0.570,0.576,0.517,0.526,0.607,0.511,0.626,0.428,0.486,0.541,2.045,Employee,Female,South
5,5,131,26.5%,57.496,2.008,2.687,0.627,0.628,0.507,0.568,0.576,0.588,0.637,0.516,0.570,0.597,2.397,Employee,Male,North
6,6,21,4.2%,53.952,1.810,2.667,0.559,0.573,0.465,0.547,0.527,0.598,0.561,0.512,0.485,0.555,2.000,Employee,Female,Center
7,7,37,7.5%,53.649,1.946,2.649,0.611,0.666,0.482,0.551,0.558,0.568,0.659,0.516,0.551,0.562,2.162,Employee,Male,Center
8,8,51,10.3%,74.804,1.882,1.824,0.461,0.489,0.223,0.420,0.646,0.343,0.636,0.254,0.363,0.382,2.098,Retired,Female,North
9,9,25,5.1%,70.880,1.720,2.280,0.507,0.470,0.242,0.371,0.545,0.341,0.614,0.328,0.344,0.423,2.000,Unemployed,Female,North



 Mixed (L2+Hamming) — k=10  α=0.20 cluster summary


,Cluster,N,%,Age,CitySize,FamilySize,Income,Wealth,Debt,FinEdu,ESG,Digital,BankFriend,LifeStyle,Luxury,Saving,Investments,Job,Gender,Area
0,0,18,3.6%,57.056,1.611,3.056,0.508,0.567,0.526,0.528,0.536,0.526,0.572,0.425,0.423,0.516,2.222,Employee,Male,South
1,1,34,6.9%,54.382,1.971,2.676,0.616,0.681,0.497,0.557,0.562,0.584,0.664,0.522,0.546,0.574,2.265,Employee,Male,Center
2,2,47,9.5%,74.936,1.638,2.340,0.451,0.428,0.232,0.410,0.607,0.367,0.685,0.354,0.332,0.374,1.872,Retired,Male,North
3,3,21,4.2%,66.952,1.714,3.000,0.572,0.575,0.514,0.526,0.602,0.524,0.619,0.437,0.480,0.549,2.095,Employee,Female,South
4,4,119,24.0%,57.420,2.000,2.445,0.593,0.612,0.499,0.551,0.579,0.581,0.616,0.515,0.518,0.591,2.269,Employee,Female,North
5,5,31,6.3%,65.161,1.935,2.129,0.520,0.517,0.279,0.393,0.564,0.377,0.586,0.380,0.379,0.417,1.871,Unemployed,Female,North
6,6,21,4.2%,53.952,1.810,2.667,0.559,0.573,0.465,0.547,0.527,0.598,0.561,0.512,0.485,0.555,2.000,Employee,Female,Center
7,7,24,4.8%,59.292,2.042,2.333,0.575,0.573,0.311,0.484,0.567,0.498,0.645,0.480,0.464,0.382,2.000,Unemployed,Male,North
8,8,51,10.3%,75.392,1.882,1.863,0.457,0.483,0.232,0.425,0.644,0.344,0.644,0.256,0.367,0.383,2.078,Retired,Female,North
9,9,129,26.1%,57.705,1.992,2.713,0.626,0.627,0.512,0.571,0.574,0.588,0.637,0.517,0.571,0.597,2.403,Employee,Male,North



 Gower — k=2 cluster summary


,Cluster,N,%,Age,CitySize,FamilySize,Income,Wealth,Debt,FinEdu,ESG,Digital,BankFriend,LifeStyle,Luxury,Saving,Investments,Job,Gender,Area
0,0,257,51.9%,60.105,1.938,2.591,0.587,0.590,0.458,0.524,0.575,0.547,0.645,0.486,0.510,0.536,2.163,Employee,Male,North
1,1,238,48.1%,63.256,1.887,2.382,0.541,0.563,0.393,0.505,0.589,0.487,0.614,0.426,0.453,0.507,2.214,Employee,Female,North


In [32]:
summary_df = (
    df_best
    .assign(Distance=df_best["num_dist"] + "+" + df_best["cat_dist"])
    [["Distance", "alpha", "k", "mean_sil", "std_sil"]]
    .rename(columns={"alpha": "Best alpha", "k": "Best k",
                     "mean_sil": "Mean silhouette", "std_sil": "Std silhouette"})
    .reset_index(drop=True)
)

summary_df = pd.concat([
    summary_df,
    pd.DataFrame([{
        "Distance":        "Gower",
        "Best alpha":      None,
        "Best k":          best_gower["k"],
        "Mean silhouette": round(best_gower["mean_sil"], 4),
        "Std silhouette":  round(best_gower["std_sil"],  4),
    }])
], ignore_index=True).sort_values("Mean silhouette", ascending=False).reset_index(drop=True)

print("\nFinal comparison — best (alpha, k) per distance metric:")
display(summary_df)

winner_row        = summary_df.iloc[0]
winner            = winner_row["Distance"]
best_alpha_winner = winner_row["Best alpha"]
winner_k          = int(winner_row["Best k"])
winner_sil        = float(winner_row["Mean silhouette"])
winner_lbls       = umap_sources[
    winner if winner == "Gower" else
    next(k for k in umap_sources if k.startswith(f"Mixed ({winner.replace('+', '+')}"))
]["labels"]

print(f"\nBest performing distance : {winner}")
if best_alpha_winner is not None:
    print(f"Optimal alpha            : {best_alpha_winner:.4f}")
print(f"Best k                   : {winner_k}")
print(f"Mean silhouette          : {winner_sil:.4f}")

# ── Save for comparison notebook ──────────────────────────────────────────────
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

pickle.dump({
    "summary_df":       summary_df,
    "winner":           winner,
    "best_alpha":       best_alpha_winner,
    "winner_k":         winner_k,
    "winner_sil":       winner_sil,
    "winner_lbls":      winner_lbls,
    "umap_sources":     umap_sources,
    "df_sweep":         df_sweep,
    "df_best":          df_best,
    "best_gower":       best_gower,
}, open(RESULTS_DIR / "weighted_results.pkl", "wb"))

logger.info("Saved weighted_results.pkl — winner=%s  alpha=%s  k=%d  sil=%.4f",
            winner, best_alpha_winner, winner_k, winner_sil)


Final comparison — best (alpha, k) per distance metric:


,Distance,Best alpha,Best k,Mean silhouette,Std silhouette
0,L1+Tanimoto,0.2,10,0.6924,0.0000
1,Canberra+Tanimoto,0.2,10,0.6798,0.0000
2,L2+Tanimoto,0.2,10,0.6715,0.0000
3,L1+Hamming,0.2,10,0.6574,0.0000
4,Canberra+Hamming,0.2,10,0.6426,0.0000
5,L2+Hamming,0.2,10,0.6286,0.0000
6,Gower,None,2,0.1774,0.0271


2026-03-19 21:02:04,210 — INFO — Saved weighted_results.pkl — winner=L1+Tanimoto  alpha=0.2  k=10  sil=0.6924



Best performing distance : L1+Tanimoto
Optimal alpha            : 0.2000
Best k                   : 10
Mean silhouette          : 0.6924
